## Reproductible fitting for the `jmstate` package

In [ ]:
pip install jmstate matplotlib pandas

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import torch

In [ ]:
torch.manual_seed(42)


TAU = 6.0


def reg(t: torch.Tensor, psi: torch.Tensor):
    b, w1, w2 = psi.chunk(3, dim=-1)  # Extract relevant terms

    # psi has shape (n_chains, n_individuals, n_repetitions)
    return (b + w1 * t + (w2 - w1) * (t > TAU) * (t - TAU)).unsqueeze(-1)


def link(t: torch.Tensor, psi: torch.Tensor):
    b, w1, w2 = psi.chunk(3, dim=-1)

    diff = (w2 - w1) * (t > TAU)
    val = b + w1 * t + diff * (t - TAU)
    der = w1 + diff
    return torch.cat([val.unsqueeze(-1), der.unsqueeze(-1)], dim=-1)


def random_far_apart(
    n: int, m: int, a: torch.Tensor, b: torch.Tensor, min_dist: torch.Tensor
):
    L_free = (b - a) - (m - 1) * min_dist

    y = torch.rand(n, m) * L_free
    y, _ = torch.sort(y, dim=1)

    gap_offset = torch.arange(m) * min_dist

    return a + y + gap_offset

In [ ]:
from jmstate.functions import Exponential, gamma_plus_b
from jmstate.typedefs import ModelDesign

# Survival model specification
surv = {
    (0, 1): (Exponential(0.1), link),
    (0, 2): (Exponential(0.01), link),
    (1, 2): (Exponential(0.2), link),
}

# Model design gathers regression, link and hazard functions
model_design = ModelDesign(gamma_plus_b, reg, surv)

In [ ]:
from jmstate.typedefs import CovParams, ModelParams

# Gaussian means
gamma = torch.tensor([2.5, -1.3, 0.2])

# Covariance matrices
Q = torch.diag(torch.tensor([0.6, 0.2, 0.3]))
R = torch.tensor([[1.7]])

# Link parameters
alphas = {
    (0, 1): torch.tensor([-0.5, -3.0]),
    (0, 2): torch.tensor([-1.0, -5.0]),
    (1, 2): torch.tensor([0.0, -1.2]),
}

# Covariate parameters
betas = {
    (0, 1): torch.tensor([-1.3]),
    (0, 2): torch.tensor([-0.9]),
    (1, 2): torch.tensor([-0.7]),
}

# Instance declaration
true_params = ModelParams(
    gamma,
    CovParams.from_cov(Q, method="diag"),
    CovParams.from_cov(R, method="ball"),
    alphas,
    betas,
)

print(true_params.as_dict)

In [ ]:
from torch.distributions import MultivariateNormal

from jmstate import MultiStateJointModel
from jmstate.typedefs import ModelData, SampleData

# Declare the true underlying model
true_model = MultiStateJointModel(model_design, true_params)


def gen_data(n: int, m: int):
    # Censoring times
    c = torch.rand(n, 1) * 5 + 10

    # Covariates
    x = torch.randn(n, 1)

    # Latent and noise distributions
    Q_dist = MultivariateNormal(torch.zeros(Q.size(0)), Q)
    R_dist = MultivariateNormal(torch.zeros(R.size(0)), R)

    # Individual effects
    b = Q_dist.sample((n,))
    psi = model_design.individual_effects_fn(gamma, x, b)

    # Generates random evaluations points with a minimum distance
    a = torch.zeros((n, 1))
    b = torch.full((n, 1), 15)
    t = random_far_apart(n, m, a, b, 0.7 * b / m)

    # Defines initial state for individuals
    trajectories_init = [[(0.0, 0)]] * n

    # Samples trajectories
    sample_data = SampleData(x, trajectories_init, psi)
    trajectories = true_model.sample_trajectories(sample_data, c)

    # Samples longitudinal values
    y = model_design.regression_fn(t, psi)
    y += R_dist.sample(y.shape[:2])

    # Censors longitudinal measurements exceeding censoring times
    y[t > c] = torch.nan

    return x, t, y, trajectories, c


# Generate data
data = ModelData(*gen_data(1000, 20))

In [ ]:
plt.plot(data.t[:50].T, data.y[:50].squeeze(-1).T)
plt.title("Longitudinal sample")
plt.xlabel("Time")
plt.ylabel("Value")
plt.tight_layout()
plt.savefig("../figures/simulated-sample.pdf")
plt.show()

In [ ]:
from jmstate.utils import build_buckets

buckets = build_buckets(data.trajectories)
print({key: val.idxs.numel() for key, val in buckets.items()})

In [ ]:
# Declares initial parameters; zero mean and unit variance
alpha_shared = torch.zeros(2)

init_params = ModelParams(
    torch.zeros_like(gamma),
    CovParams.from_cov(torch.eye(Q.size(0)), method="diag"),
    CovParams.from_cov(torch.eye(R.size(0)), method="ball"),
    {k: torch.zeros_like(v) for k, v in alphas.items()},
    {k: torch.zeros_like(v) for k, v in betas.items()},
)

init_params_shared = ModelParams(
    torch.zeros_like(gamma),
    CovParams.from_cov(torch.eye(Q.size(0)), method="diag"),
    CovParams.from_cov(torch.eye(R.size(0)), method="ball"),
    {k: alpha_shared for k, v in alphas.items()},
    {k: torch.zeros_like(v) for k, v in betas.items()},
)

init_params_full =  ModelParams(
    torch.zeros_like(gamma),
    CovParams.from_cov(torch.eye(Q.size(0)), method="full"),
    CovParams.from_cov(torch.eye(R.size(0)), method="ball"),
    {k: alpha_shared for k, v in alphas.items()},
    {k: torch.zeros_like(v) for k, v in betas.items()},
)

In [ ]:
from jmstate.jobs import LogParamsHistory, ParamStop

# Declares initial model
model = MultiStateJointModel(model_design, init_params)

# Runs optimization process
metrics = model.fit(
    data,
    job_factories=[
        ParamStop(rtol=0.1),
        LogParamsHistory(),  # Returns a list of ModelParams
    ],
    max_iterations=500,
)

In [ ]:
from jmstate.visualization import plot_params_history

plot_params_history(metrics.params_history, true_params=true_params, show=False)
plt.savefig("../figures/stochastic-optimization.pdf")
plt.show()

In [ ]:
params_list: list[torch.Tensor] = []
aics_list: list[float] = []
bics_list: list[float] = []
n_runs = 2

for _ in range(n_runs):
    data = ModelData(*gen_data(1000, 20))

    model = MultiStateJointModel(model_design, init_params)
    model.fit(
        data,
        job_factories=[ParamStop(rtol=0.1)],
        max_iterations=500,
    )
    model.compute_summary()

    model_shared = MultiStateJointModel(model_design, init_params_shared)
    model_shared.fit(
        data,
        job_factories=[ParamStop(rtol=0.1)],
        max_iterations=500,
    )
    model_shared.compute_summary()

    model_full = MultiStateJointModel(model_design, init_params_full)
    model_full.fit(
        data,
        job_factories=[ParamStop(rtol=0.1)],
        max_iterations=500,
    )
    model_full.compute_summary()

    params_list.append(model.params_.as_flat_tensor.view(1, -1))
    aics_list.append((model.aic, model_shared.aic, model_full.aic))
    bics_list.append((model.bic, model_shared.bic, model_full.bic))

In [ ]:
names = list(true_params.as_dict.keys())

stacked_params = torch.cat(params_list, dim=0)
mean = stacked_params.mean(dim=0)
std = stacked_params.std(dim=0)
rmse = (std**2 + (mean - true_model.params_.as_flat_tensor) ** 2).sqrt()

data = {
    "True value": true_model.params_.as_flat_tensor.numpy(),
    "Mean": mean.numpy(),
    "Std error": std.numpy(),
    "RMSE": rmse.numpy(),
}

index = [
    f"{names[i]}[{j}]" 
    if true_params.as_list[i].numel() > 1 
    else f"{names[i]}" 
    for i in range(len(names)) 
    for j in range(true_params.as_list[i].numel())
] 


df = pd.DataFrame(data, index=index)
pd.set_option("display.float_format", "{:.4e}".format)

print(df)